# 04 - LLM Experiments

Experiments with LLM-based risk analysis using PRISM's `LLMAnalyzer` and `RiskExtractor`.

## Objectives
- Demonstrate LLMAnalyzer for single and batch analysis
- Use RiskExtractor to structure LLM outputs
- Compare LLM outputs across risk levels
- Explore prompt engineering and response parsing

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
from src.models.llm import LLMAnalyzer
from src.models.llm.risk_extractor import RiskExtractor, RiskAnalysis

## 1. Single Project Analysis

In [ ]:
# Set your OpenAI API key (or use env OPENAI_API_KEY)
import os
api_key = os.getenv('OPENAI_API_KEY', 'your-key-here')

if api_key and api_key != 'your-key-here':
    analyzer = LLMAnalyzer(api_key=api_key, model='gpt-3.5-turbo')
    
    result = analyzer.analyze_project(
        project_name='Mobile App Phase 2',
        status_comments='Delays due to API integration issues. Team morale is low. Budget concerns emerging.'
    )
    
    print('Sentiment:', result['sentiment_score'], result['sentiment_label'])
    print('Risk Level:', result['risk_level'])
    print('Categories:', result['risk_categories'])
    print('Summary:', result['summary'])
else:
    print('Set OPENAI_API_KEY or api_key to run LLM analysis')

## 2. Batch Analysis and RiskExtractor

In [ ]:
# Load sample data
df = pd.read_csv('../data/raw/sample_projects.csv')
projects = df.head(3).to_dict(orient='records')

if api_key and api_key != 'your-key-here':
    results = analyzer.analyze_batch(projects, text_field='status_comments', name_field='project_name')
    
    extractor = RiskExtractor()
    analyses = extractor.extract(results)
    llm_df = extractor.to_dataframe()
    
    print(llm_df[['project_name', 'sentiment_score', 'risk_level', 'summary']])
    print('\nSummary stats:', extractor.get_summary_stats())
else:
    print('Set OPENAI_API_KEY to run batch analysis')

## 3. Risk Categories Distribution

In [ ]:
if api_key and api_key != 'your-key-here' and 'extractor' in dir() and extractor.analyses:
    stats = extractor.get_summary_stats()
    print('Risk distribution:', stats.get('risk_distribution', {}))
    print('Category counts:', stats.get('category_counts', {}))
    print('Avg sentiment:', stats.get('avg_sentiment', 0))
else:
    print('Run batch analysis above first')